In [1]:
# !apt -yq install ffmpeg >/dev/null
!pip install "transformers>=4.44" torch torchaudio soundfile tqdm >/dev/null

In [ ]:
# ================================
# Massive Whisper inference runner
# One JSONL per meeting, continuous append
# ================================

import os, re, json, time, hashlib, logging
from datetime import datetime, timezone
from collections import defaultdict
from pathlib import Path

import torch
import soundfile as sf
from transformers import pipeline
from tqdm import tqdm

# ---------- CONFIG ----------
INPUT_DIR = Path("/work/FPSC/wav_speeches")      # folder with 89k wavs
OUTPUT_DIR = Path("/work/FPSC/output3")          # per-meeting JSONL outputs
LOG_DIR = Path("/work/FPSC/output3")             # log file lives here

# Choose your model here
MODEL = "davidilag/whisper-large-no-is-fo-100h-30k-steps"
# MODEL = "carlosdanielhernandezmena/whisper-large-faroese-8k-steps-100h"

# Force language & task (Faroese corpus)
MODEL_LANG = "fo"     # 'fo' is Faroese; use 'is' for Icelandic if needed
TASK = "transcribe"

# Inference settings
CHUNK_LEN = 30
STRIDE_LEN = 5
TEMPERATURE = 0.0
NUM_BEAMS = 1
# -----------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

def short_model_name(model: str) -> str:
    mapping = {
        "davidilag/whisper-large-no-is-fo-100h-30k-steps": "whisper-no-is-fo",
        "carlosdanielhernandezmena/whisper-large-faroese-8k-steps-100h": "whisper-fo",
    }
    if model in mapping:
        return mapping[model]
    tail = model.split("/")[-1]
    tail = tail.replace("whisper-large-", "whisper-")
    tail = re.sub(r"[^a-z0-9\-]+", "-", tail.lower())
    return tail[:48].strip("-")

SHORT = short_model_name(MODEL)

# ---------- Resume control (find last processed meeting & delete it) ---------- 
resume_meeting_id = None                                                           
out_pat = re.compile(rf"^M(\d+){re.escape('_'+SHORT)}\.jsonl$", re.IGNORECASE)      
existing_outputs = []                                                              
for p in OUTPUT_DIR.iterdir():                                                    
    m = out_pat.match(p.name)                                                      
    if m:                                                                           
        existing_outputs.append((int(m.group(1)), p))                              
                                                                                    
if existing_outputs:                                                               
    existing_outputs.sort(key=lambda t: t[0])                                       
    last_num, last_file = existing_outputs[-1]                                      
    resume_meeting_id = f"M{last_num}"                                             
    # Delete the last processed meeting file so we can redo it cleanly             
    try:                                                                            
        last_file.unlink()                                                        
        print(f"⏪ Resuming from {resume_meeting_id}: deleted {last_file.name}")     
    except Exception as e:                                                          
        print(f"⚠️ Could not delete {last_file.name}: {e}")                        
else:                                                                               
    print("ℹ️ No prior outputs found for this model tag; starting fresh.")          
# -----------------------------------------------------------------------------  

# ---------- Logging (reuse the same latest log file if present) ----------
# Look for an existing log for this SHORT tag and append; otherwise create a new one
existing_logs = sorted(LOG_DIR.glob(f"{SHORT}_*.log"))                              
if existing_logs:                                                                   
    log_path = existing_logs[-1]                                                  
    log_mode = "a"                                                                 
    print(f"📝 Appending to existing log: {log_path}")                             
else:                                                                              
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")                                  
    log_path = LOG_DIR / f"{SHORT}_{ts}.log"                                       
    log_mode = "w"                                                               
    print(f"📝 Creating new log: {log_path}")                                      

logger = logging.getLogger("fpsc_infer")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fh = logging.FileHandler(log_path, mode=log_mode, encoding="utf-8")  # <<<
fh.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
fh.setFormatter(fmt)
ch.setFormatter(fmt)

logger.addHandler(fh)
logger.addHandler(ch)

logger.info(f"=== START/RESUME RUN ===")                                          
logger.info(f"Model: {MODEL}  (short: {SHORT})")
logger.info(f"Input: {INPUT_DIR}")
logger.info(f"Outputs: {OUTPUT_DIR}")
logger.info(f"Log file: {log_path}")
if resume_meeting_id:
    logger.info(f"Resuming from (re-doing) meeting: {resume_meeting_id}")           
else:
    logger.info("No prior outputs detected for this tag; fresh start.")         
# ---------------------------------------------

# ---------- Helpers ----------
rx = re.compile(r"^(M\d+)_S(\d{4})\.wav$", re.IGNORECASE)

def parse_ids(fname: str):
    m = rx.match(fname)
    if not m:
        return None, None, None
    meeting = m.group(1).upper()                # e.g., M1
    speech = f"S{m.group(2)}"                   # e.g., S0001
    audio_id = f"{meeting}_{speech}"            # e.g., M1_S0001
    return meeting, speech, audio_id

def sha256_of_file(path: Path, chunk=1024*1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def audio_info_sf(path: Path):
    with sf.SoundFile(str(path)) as f:
        frames = len(f)
        sr = f.samplerate
        ch = f.channels
        dur = frames / float(sr) if sr else 0.0
        subtype = f.subtype
        fmt = f.format
    return {"sample_rate_hz": sr, "channels": ch, "duration_s": dur, "subtype": subtype, "format": fmt}

def fmt_hhmmss_ms(seconds: float) -> str | None:
    if seconds is None:
        return None
    ms = int(round((seconds - int(seconds)) * 1000))
    s = int(seconds) % 60
    m = (int(seconds) // 60) % 60
    h = int(seconds) // 3600
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

# ---------- Discover files & group by meeting ----------
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")

all_wavs = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() == ".wav" and rx.match(p.name)]
if not all_wavs:
    raise RuntimeError(f"No matching WAV files found in {INPUT_DIR} (expected names like M1_S0001.wav)")

meetings = defaultdict(list)
for p in all_wavs:
    meeting, speech, audio_id = parse_ids(p.name)
    if meeting:
        meetings[meeting].append((speech, audio_id, p))

# sort speeches within each meeting (S0001 < S0002 ...)
for m in meetings:
    meetings[m].sort(key=lambda t: t[0])

total_files = sum(len(v) for v in meetings.values())
logger.info(f"Found {len(meetings)} meetings, {total_files} wav files.")

# Precompute existing completed meetings (after deletion above)                        
done_meetings = set()                                                                
for p in OUTPUT_DIR.iterdir():                                                        
    m = out_pat.match(p.name)                                                         
    if m:                                                                           
        done_meetings.add(f"M{int(m.group(1))}")                                   

# ---------- Prepare ASR pipeline ----------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

logger.info(f"Device: {device}  torch_dtype: {torch_dtype}")
asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL,
    device=device,
    torch_dtype=torch_dtype,
    chunk_length_s=CHUNK_LEN,
    stride_length_s=STRIDE_LEN,
)

# ---------- Run ----------
files_ok = 0
files_fail = 0
sum_audio_sec = 0.0
sum_wall_sec = 0.0

for meeting_id, items in sorted(meetings.items(), key=lambda kv: int(kv[0][1:])):  # sort by numeric part of M#
    out_path = OUTPUT_DIR / f"{meeting_id}_{SHORT}.jsonl"

    # Skip meetings that are already done (file exists) EXCEPT the one we just deleted            
    if meeting_id in done_meetings:                                                              
        logger.info(f"=== Skipping {meeting_id} (already completed)")                            
        continue                                                                                  
                                                                                                  
    n_items = len(items)
    print(f"=== Meeting {meeting_id} | {n_items} speeches -> {out_path}")
    logger.info(f"=== Meeting {meeting_id} | {n_items} speeches -> {out_path}")

    with out_path.open("a", encoding="utf-8") as fout:
        for speech_id, audio_id, wav_path in tqdm(items, desc=f"{meeting_id}", leave=False):
            try:
                meta = audio_info_sf(wav_path)
                dur = float(meta.get("duration_s") or 0.0)
                sum_audio_sec += dur

                # time single inference
                t0 = time.time()
                try:
                    result = asr(
                        str(wav_path),
                        return_timestamps="word",
                        generate_kwargs={
                            "language": MODEL_LANG,
                            "task": TASK,
                            "temperature": TEMPERATURE,
                            "condition_on_prev_tokens": False,
                            "num_beams": NUM_BEAMS,
                        },
                    )
                    used_word_ts = True
                except ValueError as e:
                    if "alignment_heads" in str(e) or "word" in str(e).lower():
                        logger.warning(f"{audio_id}: no word-level timestamps, falling back to segment timestamps. ({e})")
                        result = asr(
                            str(wav_path),
                            return_timestamps=True,   # segments only
                            generate_kwargs={
                                "language": MODEL_LANG,
                                "task": TASK,
                                "temperature": TEMPERATURE,
                                "condition_on_prev_tokens": False,
                                "num_beams": NUM_BEAMS,
                            },
                        )
                        used_word_ts = False
                    else:
                        raise

                t1 = time.time()
                wall_s = t1 - t0
                sum_wall_sec += wall_s
                rtf = wall_s / dur if dur > 0 else float("inf")

                chunks = result.get("chunks") or []
                words = []
                if used_word_ts:
                    for ch in chunks:
                        ts = ch.get("timestamp")
                        if not ts or ts[0] is None or ts[1] is None:
                            continue
                        wtxt = (ch.get("text") or "").strip()
                        if not wtxt:
                            continue
                        words.append({
                            "text": wtxt,
                            "start_s": float(ts[0]),
                            "end_s": float(ts[1]),
                            "start_hhmmss": fmt_hhmmss_ms(float(ts[0])),
                            "end_hhmmss": fmt_hhmmss_ms(float(ts[1])),
                        })

                record = {
                    "schema_version": "1.0",
                    "created_utc": datetime.now(timezone.utc).isoformat(),
                    "framework": "transformers",
                    "model": MODEL,
                    "audio_id": audio_id,
                    "inference_device": device,
                    "decode_params": {
                        "chunk_length_s": CHUNK_LEN,
                        "stride_length_s": STRIDE_LEN,
                        "return_timestamps": "word" if used_word_ts else "segment",
                        "temperature": TEMPERATURE,
                        "condition_on_prev_tokens": False,
                        "task": TASK,
                        "num_beams": NUM_BEAMS,
                        "language": MODEL_LANG,
                    },
                    "timing": {
                        "wall_time_s": wall_s,
                        "audio_duration_s": dur,
                        "rtf": rtf,
                    },
                    "audio": {
                        "source_path": str(wav_path),
                        "source_sha256": sha256_of_file(wav_path),
                        "source_meta": meta,
                    },
                    "transcript": {
                        "full_text": (result.get("text") or "").strip(),
                        "num_words": len(words),
                    },
                    "words": words,
                }

                fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                fout.flush()
                files_ok += 1

            except Exception as ex:
                files_fail += 1
                logger.exception(f"ERROR processing {audio_id} ({wav_path}): {ex}")

# ---------- Summary ----------
meetings_cnt = len(meetings)
audio_hours = sum_audio_sec / 3600.0
avg_rtf = (sum_wall_sec / sum_audio_sec) if sum_audio_sec > 0 else float("inf")

summary = {
    "meetings_processed": meetings_cnt,
    "files_total": files_ok + files_fail,
    "files_ok": files_ok,
    "files_fail": files_fail,
    "audio_time_hours": round(audio_hours, 2),
    "wall_time_seconds_sum": round(sum_wall_sec, 2),
    "average_RTF": round(avg_rtf, 3),
    "model": MODEL,
    "short_model": SHORT,
    "outputs_dir": str(OUTPUT_DIR),
    "log_file": str(log_path),
}
logger.info("========== RUN COMPLETE ==========")
for k, v in summary.items():
    logger.info(f"{k}: {v}")

print("\n===== SUMMARY =====")
for k, v in summary.items():
    print(f"{k}: {v}")


2025-10-17 12:31:18,504 | INFO | === START/RESUME RUN ===
2025-10-17 12:31:18,505 | INFO | Model: davidilag/whisper-large-no-is-fo-100h-30k-steps  (short: whisper-no-is-fo)
2025-10-17 12:31:18,505 | INFO | Input: /work/FPSC/wav_speeches
2025-10-17 12:31:18,506 | INFO | Outputs: /work/FPSC/output3
2025-10-17 12:31:18,506 | INFO | Log file: /work/FPSC/output3/whisper-no-is-fo_20251011-145934.log
2025-10-17 12:31:18,507 | INFO | Resuming from (re-doing) meeting: M6749


⏪ Resuming from M6749: deleted M6749_whisper-no-is-fo.jsonl
📝 Appending to existing log: /work/FPSC/output3/whisper-no-is-fo_20251011-145934.log


2025-10-17 12:31:22,433 | INFO | Found 368 meetings, 89486 wav files.
2025-10-17 12:31:22,840 | INFO | Device: cuda:0  torch_dtype: torch.float16


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

M7221:  99%|█████████▉| 121/122 [13:28<00:02,  2.67s/it]Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
2025-10-21 15:57:45,001 | INFO | === Meeting M7222 | 5 speeches -> /work/FPSC/output3/M7222_whisper-no-is-fo.jsonl


=== Meeting M7222 | 5 speeches -> /work/FPSC/output3/M7222_whisper-no-is-fo.jsonl


M7222:  80%|████████  | 4/5 [00:48<00:13, 13.13s/it]Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
2025-10-21 15:58:40,368 | INFO | === Meeting M7227 | 556 speeches -> /work/FPSC/output3/M7227_whisper-no-is-fo.jsonl


=== Meeting M7227 | 556 speeches -> /work/FPSC/output3/M7227_whisper-no-is-fo.jsonl


M7227:   0%|          | 2/556 [00:21<1:29:13,  9.66s/it]